# SMART STUDY ASSISTANT — YOLO kantukv2 (Colab, runtime GPU)
Audit → training → evaluasi. **Hanya dataset kantukv2.** Upload folder `scripts/` ke `MyDrive/smart-study-assistant/` sebelum menjalankan.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/smart-study-assistant
!pip -q install ultralytics roboflow

## 1. Unduh dataset (satu versi; catat versinya)
Isi API key Roboflow **milik Anda** lewat Colab Secrets (`ROBOFLOW_API_KEY`), jangan ditulis di notebook.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
VERSION = 1  # <-- kunci versi dataset yang dipakai
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace('testproject-8jvqn').project('kantukv2').version(VERSION).download('yolov11', location='datasets/kantukv2')
print(ds.location)

## 2. Audit (cek kelas, distribusi, duplikat/kebocoran antar split)
Lihat `metadata/audit_samples.jpg` dan perbaiki split dari gambar kantukv2 yang sama bila ada kebocoran. Isi lisensi sesuai halaman dataset.

In [ ]:
!python scripts/audit_dataset.py datasets/kantukv2 --version {VERSION} --license 'ISI-DARI-ROBOFLOW'

## 3. Fine-tuning (yolo11n pretrained)

In [ ]:
!python scripts/train_yolo.py datasets/kantukv2/data.yaml --model yolo11n.pt --epochs 50 --imgsz 640 --batch 16 --device 0

## 4. Evaluasi pada test split + latensi
Hasil: `metadata/eval_report.json`, confusion matrix. Bobot: `models/best.pt` (salin ke `models/` di repo lokal).

In [ ]:
!python scripts/evaluate_yolo.py datasets/kantukv2/data.yaml --weights models/best.pt --device 0

In [ ]:
from IPython.display import Image, display
import glob
for f in glob.glob('metadata/confusion_matrix*.png'): display(Image(f))